# Day 07 — Uzaklık ve Benzerlik Yöntemleri
## Öklid, Manhattan, Kosinüs Benzerliği ve K-En Yakın Komşu (k-NN) Mantığı

> **Aşama:** Faz 1 — Problem, Veri ve Geliştirme Temelleri (Day 01–08)
> **Resmi Staj Defteri Konusu:** Uzaklık ve Benzerlik Yöntemleri (Yaprak 13 & 14)

### 1. Problem
İki halı deseni, iki sensör telemetri anı veya iki dokuma deseni özniteliğinin ne kadar 'benzer' olduğunu sayısal olarak belirlemek için doğru geometrik veya açısal mesafe metriğini seçmek gerekir. Yanlış metrik seçimi (örneğin büyüklüğe duyarlı verilerde kosinüs veya yoğun ölçek farkı olan verilerde Manhattan kullanımı) hatalı eşleşmelere yol açar.

### 2. Why the Problem Matters
Görsel arama motorları, k-NN sınıflandırıcılar ve öneri sistemleri tamamen vektörel uzaklık matrisleri üzerinde çalışır. Mesafelerin özellikleri (simetri, üçgen eşitsizliği) bilinmeden model kurulamaz.

### 3. Engineering Concepts
- **Öklid Mesafesi ($L_2$)**: Doğrudan geometrik mesafe: $d_2(u, v) = \sqrt{\sum (u_i - v_i)^2}$.
- **Manhattan Mesafesi ($L_1$)**: Şehir içi blok mesafesi: $d_1(u, v) = \sum |u_i - v_i|$.
- **Kosinüs Benzerliği**: İki vektör arasındaki açının kosinüsü: $\cos(\theta) = \frac{u \cdot v}{\|u\| \|v\|}$.
- **k-En Yakın Komşu (k-NN)**: Sorgu vektörüne en küçük mesafedeki k adet örneğin getirilmesi.

In [ ]:
# 4. Library / API Investigation & Standalone Definitions
import numpy as np
from typing import List, Tuple

def euclidean_distance(u: np.ndarray, v: np.ndarray) -> float:
    return float(np.linalg.norm(u - v))

def manhattan_distance(u: np.ndarray, v: np.ndarray) -> float:
    return float(np.sum(np.abs(u - v)))

def cosine_similarity(u: np.ndarray, v: np.ndarray) -> float:
    nu = np.linalg.norm(u)
    nv = np.linalg.norm(v)
    if nu == 0 or nv == 0:
        return 0.0
    return float(np.dot(u, v) / (nu * nv))

def compute_pairwise_distances(features: np.ndarray, metric: str = "euclidean") -> np.ndarray:
    n = features.shape[0]
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if metric == "euclidean":
                D[i, j] = euclidean_distance(features[i], features[j])
            else:
                D[i, j] = 1.0 - cosine_similarity(features[i], features[j])
    return D

class KNNPatternMatcher:
    def __init__(self, metric: str = "euclidean"):
        self.metric = metric
        self.catalog_ids = []
        self.features = None

    def fit(self, catalog_ids: List[str], features: np.ndarray):
        self.catalog_ids = catalog_ids
        self.features = features

    def query(self, q: np.ndarray, k: int = 3) -> List[Tuple[str, float]]:
        if self.features is None or len(self.features) == 0:
            raise ValueError("Katalog veritabanı boş veya eğitilmedi.")
        scores = []
        for i, f in enumerate(self.features):
            if self.metric == "euclidean":
                dist = euclidean_distance(q, f)
            else:
                dist = 1.0 - cosine_similarity(q, f)
            scores.append((self.catalog_ids[i], dist))
        scores.sort(key=lambda x: x[1])
        return scores[:k]

u = np.array([2.0, 3.0, 5.0])
v = np.array([5.0, 7.0, 5.0])
print(f"Öklid: {euclidean_distance(u, v):.4f}")
print(f"Manhattan: {manhattan_distance(u, v):.4f}")
print(f"Kosinüs Benzerliği: {cosine_similarity(u, v):.4f}")


In [ ]:
# 5. Minimal Implementation: k-NN Matcher
matcher = KNNPatternMatcher(metric="euclidean")
catalog_ids = ["Halı_A (Kırmızı)", "Halı_B (Mavi)", "Halı_C (Yeşil)"]
features = np.array([
    [0.9, 0.1, 0.0],
    [0.0, 0.1, 0.9],
    [0.1, 0.8, 0.1]
])
matcher.fit(catalog_ids, features)

query = np.array([0.85, 0.15, 0.05])
matches = matcher.query(query, k=2)
print("En Yakın 2 Halı Eşleşmesi:", matches)

In [ ]:
# 6. Experiment: Mesafe Matrisi Hesaplama
# compute_pairwise_distances onceden tanimlandi
D = compute_pairwise_distances(features, metric="euclidean")
print("Çiftler Arası Mesafe Matrisi:\n", np.round(D, 3))



In [ ]:
# 7. Visualization: Mesafe Isı Haritası
import matplotlib.pyplot as plt

plt.figure(figsize=(5, 4))
plt.imshow(D, cmap="viridis", interpolation="nearest")
plt.colorbar(label="Öklid Mesafesi")
plt.xticks(range(3), catalog_ids, rotation=15)
plt.yticks(range(3), catalog_ids)
plt.title("Örnek Halı Desenleri Mesafe Matrisi")
plt.tight_layout()
plt.show()

In [ ]:
# 8. Validation
assert matches[0][0] == "Halı_A (Kırmızı)"
assert D[0, 0] == 0.0
assert np.allclose(D, D.T), "Mesafe matrisi simetrik olmalıdır!"
print("k-NN eşleme ve simetri doğrulamaları başarılı.")

In [ ]:
# 9. Failure Cases: Boş veritabanı sorgusu
empty_matcher = KNNPatternMatcher()
try:
    empty_matcher.query(np.array([1, 2]))
except ValueError as e:
    print("Beklenen boş eşleyici hatası yakalandı:", e)

### 10. Conclusions
Uzaklık ve benzerlik yöntemleri (Öklid, Manhattan, Kosinüs) incelenmiş, k-NN yaklaşımıyla en yakın desen arama mantığı doğrulanmıştır.